# Self-Interacting Neutrinos (SINu) — Massive Neutrino Case

This notebook demonstrates the effect of neutrino self-interactions on the CMB power spectra
and matter power spectrum for a cosmology with **one massive neutrino species** ($m_\nu = 0.06$ eV)
alongside two massless species.

In the massive case, both the massive (ncdm) and massless (ur) neutrinos participate in
self-interactions. The collision terms in the Boltzmann hierarchy are computed using
momentum-dependent tables (`Coll_integrals_5_qbins.dat`) that account for the non-trivial
phase-space distribution of massive neutrinos.

We also show the interplay between neutrino mass and self-interaction: both effects suppress
small-scale power, but through different mechanisms (mass via free-streaming suppression after
neutrinos become non-relativistic; SINu via isotropisation of the distribution function).

Reference: Camarena, Fogli, Lesgourgues, Smirnov, [arXiv:2309.03941](https://arxiv.org/abs/2309.03941)

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
from classy import Class

plt.rcParams.update({'font.size': 13, 'figure.dpi': 100})

## Cosmological parameters

We use Planck 2018 best-fit parameters with one massive neutrino ($m_\nu = 0.06$ eV,
T_ncdm = 0.71611, N_ur = 2.0328) and compare vanilla vs SINu-enabled runs.

In [ ]:
# Common cosmological parameters (Planck 2018 best-fit, one massive neutrino)
common = {
    'output': 'tCl,pCl,lCl,mPk',
    'lensing': 'yes',
    'gauge': 'synchronous',
    'h': 0.6732,
    'omega_b': 0.022383,
    'omega_cdm': 0.12011,
    'A_s': 2.1005e-9,
    'n_s': 0.96605,
    'tau_reio': 0.0543,
    'N_ur': 2.0328,     # two massless neutrino species
    'N_ncdm': 1,        # one massive species
    'm_ncdm': 0.06,     # mass in eV
    'T_ncdm': 0.71611,  # temperature ratio T_ncdm/T_cmb
    'P_k_max_h/Mpc': 1.0,
    'z_pk': 0,
    'l_max_scalars': 2500,
}

# Cases to compare
cases = [
    {'label': 'SINu off (vanilla)',    'log10_G': None,  'color': 'k',        'ls': '-'},
    {'label': r'$\log_{10}G_\mathrm{eff}=-2.0$', 'log10_G': -2.0, 'color': 'royalblue', 'ls': '--'},
    {'label': r'$\log_{10}G_\mathrm{eff}=-1.5$', 'log10_G': -1.5, 'color': 'tomato',    'ls': '-.'},
    {'label': r'$\log_{10}G_\mathrm{eff}=-1.0$', 'log10_G': -1.0, 'color': 'seagreen',  'ls': ':'},
]

## Run CLASS for each case

In [ ]:
results = []

for case in cases:
    params = dict(common)
    if case['log10_G'] is not None:
        params['log10_G_eff_nu'] = case['log10_G']
        params['interacting_nu'] = 1

    print(f"Computing: {case['label']} ...", end=' ', flush=True)
    cosmo = Class()
    cosmo.set(params)
    cosmo.compute()

    h = cosmo.h()
    kvec = np.logspace(-3, 0, 500)          # k in h/Mpc
    pk   = cosmo.get_pk_all(kvec * h, 0.) * h**3  # P(k) in (Mpc/h)^3

    cls  = cosmo.lensed_cl(2500)
    ell  = np.array(cls['ell'][2:], dtype=float)
    cl_tt = np.array(cls['tt'][2:]) * ell * (ell + 1) / (2 * np.pi)
    cl_ee = np.array(cls['ee'][2:]) * ell * (ell + 1) / (2 * np.pi)

    results.append({'kvec': kvec, 'pk': pk, 'ell': ell,
                    'cl_tt': cl_tt, 'cl_ee': cl_ee, **case})

    cosmo.struct_cleanup()
    cosmo.empty()
    print('done')

## CMB Temperature Power Spectrum $C_\ell^{TT}$

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

vanilla = results[0]

for r in results:
    axes[0].plot(r['ell'], r['cl_tt'], color=r['color'], ls=r['ls'], label=r['label'])
    axes[1].plot(r['ell'], r['cl_tt'] / vanilla['cl_tt'],
                 color=r['color'], ls=r['ls'], label=r['label'])

axes[0].set_xscale('log')
axes[0].set_xlabel(r'Multipole $\ell$')
axes[0].set_ylabel(r'$D_\ell^{TT}$ (dimensionless units)')
axes[0].set_title(r'CMB TT power spectrum ($m_\nu=0.06$ eV)')
axes[0].legend(fontsize=11)
axes[0].set_xlim(2, 2500)

axes[1].axhline(1, color='k', lw=0.8, ls='--')
axes[1].set_xscale('log')
axes[1].set_xlabel(r'Multipole $\ell$')
axes[1].set_ylabel(r'$D_\ell^{TT}\,(\mathrm{SINu})/D_\ell^{TT}\,(\mathrm{vanilla})$')
axes[1].set_title('Ratio to vanilla')
axes[1].legend(fontsize=11)
axes[1].set_xlim(2, 2500)

fig.tight_layout()
plt.show()

## CMB E-mode Polarisation Power Spectrum $C_\ell^{EE}$

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for r in results:
    axes[0].plot(r['ell'], r['cl_ee'], color=r['color'], ls=r['ls'], label=r['label'])
    axes[1].plot(r['ell'], r['cl_ee'] / vanilla['cl_ee'],
                 color=r['color'], ls=r['ls'], label=r['label'])

axes[0].set_xscale('log')
axes[0].set_xlabel(r'Multipole $\ell$')
axes[0].set_ylabel(r'$D_\ell^{EE}$ (dimensionless units)')
axes[0].set_title(r'CMB EE power spectrum ($m_\nu=0.06$ eV)')
axes[0].legend(fontsize=11)
axes[0].set_xlim(2, 2500)

axes[1].axhline(1, color='k', lw=0.8, ls='--')
axes[1].set_xscale('log')
axes[1].set_xlabel(r'Multipole $\ell$')
axes[1].set_ylabel(r'$D_\ell^{EE}\,(\mathrm{SINu})/D_\ell^{EE}\,(\mathrm{vanilla})$')
axes[1].set_title('Ratio to vanilla')
axes[1].legend(fontsize=11)
axes[1].set_xlim(2, 2500)

fig.tight_layout()
plt.show()

## Matter Power Spectrum $P(k)$

Neutrino self-interactions suppress free-streaming, partially countering the neutrino mass
suppression of $P(k)$ at small scales (large $k$).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for r in results:
    axes[0].loglog(r['kvec'], r['pk'], color=r['color'], ls=r['ls'], label=r['label'])
    axes[1].semilogx(r['kvec'], r['pk'] / vanilla['pk'],
                     color=r['color'], ls=r['ls'], label=r['label'])

axes[0].set_xlabel(r'$k\;[h/\mathrm{Mpc}]$')
axes[0].set_ylabel(r'$P(k)\;[(\mathrm{Mpc}/h)^3]$')
axes[0].set_title(r'Matter power spectrum ($m_\nu=0.06$ eV)')
axes[0].legend(fontsize=11)

axes[1].axhline(1, color='k', lw=0.8, ls='--')
axes[1].set_xlabel(r'$k\;[h/\mathrm{Mpc}]$')
axes[1].set_ylabel(r'$P(k)\,(\mathrm{SINu})\,/\,P(k)\,(\mathrm{vanilla})$')
axes[1].set_title('Ratio to vanilla')
axes[1].legend(fontsize=11)

fig.tight_layout()
plt.show()

## Comparison: mass effect vs SINu effect on P(k)

To isolate the contribution of each effect, we also run a massless case with SINu
and compare all four combinations: (no mass, no SINu), (mass, no SINu), (no mass, SINu), (mass, SINu).

In [ ]:
# Run the four combinations for log10_G_eff_nu = -1.5
combos = [
    {'label': 'Massless, SINu off',  'N_ur': 3.044,  'N_ncdm': 0, 'sinu': False, 'color': 'k',        'ls': '-'},
    {'label': 'Massive, SINu off',   'N_ur': 2.0328, 'N_ncdm': 1, 'sinu': False, 'color': 'slategray', 'ls': '--'},
    {'label': 'Massless, SINu on',   'N_ur': 3.044,  'N_ncdm': 0, 'sinu': True,  'color': 'tomato',   'ls': '-.'},
    {'label': 'Massive, SINu on',    'N_ur': 2.0328, 'N_ncdm': 1, 'sinu': True,  'color': 'steelblue','ls': ':'},
]

base_params = {
    'output': 'mPk', 'gauge': 'synchronous',
    'h': 0.6732, 'omega_b': 0.022383, 'omega_cdm': 0.12011,
    'A_s': 2.1005e-9, 'n_s': 0.96605, 'tau_reio': 0.0543,
    'm_ncdm': 0.06, 'T_ncdm': 0.71611,
    'P_k_max_h/Mpc': 1.0, 'z_pk': 0,
}

combo_results = []
kvec = np.logspace(-3, 0, 500)

for c in combos:
    p = dict(base_params)
    p['N_ur']   = c['N_ur']
    p['N_ncdm'] = c['N_ncdm']
    if c['N_ncdm'] == 0:
        p.pop('m_ncdm', None)
        p.pop('T_ncdm', None)
    if c['sinu']:
        p['log10_G_eff_nu'] = -1.5
        p['interacting_nu'] = 1

    print(f"Computing: {c['label']} ...", end=' ', flush=True)
    cosmo = Class()
    cosmo.set(p)
    cosmo.compute()
    h = cosmo.h()
    pk = cosmo.get_pk_all(kvec * h, 0.) * h**3
    combo_results.append({'pk': pk, **c})
    cosmo.struct_cleanup()
    cosmo.empty()
    print('done')

# Reference: massless, SINu off
pk_ref = combo_results[0]['pk']

fig, ax = plt.subplots(figsize=(8, 5))
for r in combo_results:
    ax.semilogx(kvec, r['pk'] / pk_ref, color=r['color'], ls=r['ls'], label=r['label'])

ax.axhline(1, color='k', lw=0.6, ls='--')
ax.set_xlabel(r'$k\;[h/\mathrm{Mpc}]$')
ax.set_ylabel(r'$P(k)\,/\,P(k)[\mathrm{massless,\,SINu\,off}]$')
ax.set_title(r'Mass vs SINu effect on $P(k)$ ($\log_{10}G_\mathrm{eff}=-1.5$)')
ax.legend(fontsize=11)
fig.tight_layout()
plt.show()

## Summary

- **Neutrino mass** suppresses $P(k)$ on small scales (below the free-streaming scale
  $k_\mathrm{fs} \sim 0.02\,(m_\nu/0.1\,\mathrm{eV})\,h/\mathrm{Mpc}$).
- **SINu** isotropises the neutrino distribution, reducing anisotropic stress and partially
  counteracting the free-streaming suppression. The combination (massive + SINu) lies between
  the two individual effects.
- In the CMB, both mass and SINu alter peak positions and heights, but at different angular scales.
- To enable SINu for a massive neutrino cosmology, simply set `log10_G_eff_nu` (and optionally
  `interacting_nu = 1`). The same coupling applies to all neutrino species (ur + ncdm).